In [10]:
# ==========================================================
# 📦 Install
# ==========================================================
!pip install gradio requests pandas openpyxl beautifulsoup4 > /dev/null

# ==========================================================
# 📥 Imports
# ==========================================================
import gradio as gr
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup
from datetime import datetime
from functools import lru_cache

# ==========================================================
# 📁 Input Files
# ==========================================================
hec_excel_path = "/content/drive/MyDrive/UMT Research Portal Automation Project/List of National Journals 2024-25.xlsx"
impact_excel_path = "/content/drive/MyDrive/UMT Research Portal Automation Project/IF2025 JCR.xlsx"
export_path = "/content/submitted_metadata.xlsx"

# ==========================================================
# 📊 Load Impact Factor Excel
# ==========================================================
impact_df = pd.read_excel(impact_excel_path)
impact_df["ISSN"] = impact_df["ISSN"].astype(str).str.strip()
impact_df["eISSN"] = impact_df["eISSN"].astype(str).str.strip()

# ==========================================================
# 📚 Load HEC Journal List From Excel
# ==========================================================
def load_hec_journals():
    """
    Load HEC journals from Excel file.
    Uses both ISSN P and ISSN E columns.
    """
    df = pd.read_excel(hec_excel_path)

    # Print columns to help debug (optional)
    print("Available columns in HEC Excel:", df.columns.tolist())

    # Use the exact column names from the file
    issn_p_col = "ISSN P"
    issn_e_col = "ISSN E"
    category_col = "Category"

    # Create two dataframes - one for Print ISSN, one for Electronic ISSN
    hec_data = []

    for _, row in df.iterrows():
        # Extract just the letter from "Y category" -> "Y"
        cat_value = str(row[category_col]).strip().upper()
        # Extract first letter (X, Y, or W)
        category = cat_value[0] if cat_value and cat_value[0] in ['X', 'Y', 'W'] else 'W'

        # Add Print ISSN
        if pd.notna(row[issn_p_col]):
            issn_p = str(row[issn_p_col]).strip()
            if issn_p and issn_p != 'nan':
                hec_data.append({"ISSN": issn_p, "HEC Category": category})

        # Add Electronic ISSN
        if pd.notna(row[issn_e_col]):
            issn_e = str(row[issn_e_col]).strip()
            if issn_e and issn_e != 'nan':
                hec_data.append({"ISSN": issn_e, "HEC Category": category})

    hec_df = pd.DataFrame(hec_data)
    print(f"Loaded {len(hec_df)} ISSN entries from HEC file")

    return hec_df

hec_df = load_hec_journals()

# ==========================================================
# 🧹 Helpers
# ==========================================================
def clean_html(html):
    return BeautifulSoup(html, "html.parser").get_text(" ", strip=True) if html else None

def extract_doi(text):
    match = re.search(r"10\.\d{4,9}/[-._;()/:A-Z0-9]+", text, re.I)
    return match.group(0) if match else text.strip()

def format_publication_date(date_parts):
    """
    Convert Crossref date-parts to human-readable format.
    date_parts is typically [[year, month, day]] or [[year, month]] or [[year]]
    Returns string like "November 2020" or "2020-11-15" or just "2020"
    """
    if not date_parts or not date_parts[0]:
        return None

    parts = date_parts[0]

    if len(parts) == 1:
        # Only year
        return str(parts[0])
    elif len(parts) == 2:
        # Year and month
        year, month = parts[0], parts[1]
        month_names = ["January", "February", "March", "April", "May", "June",
                       "July", "August", "September", "October", "November", "December"]
        return f"{month_names[month-1]} {year}"
    elif len(parts) >= 3:
        # Year, month, and day
        year, month, day = parts[0], parts[1], parts[2]
        month_names = ["January", "February", "March", "April", "May", "June",
                       "July", "August", "September", "October", "November", "December"]
        return f"{month_names[month-1]} {day}, {year}"

    return str(parts[0])

# ==========================================================
# 🔎 Cached API Calls
# ==========================================================
@lru_cache(maxsize=256)
def fetch_crossref(doi):
    try:
        r = requests.get(f"https://api.crossref.org/works/{doi}", timeout=7)
        if r.status_code == 200:
            return r.json().get("message")
    except:
        pass
    return None

@lru_cache(maxsize=256)
def fetch_openalex(doi):
    try:
        r = requests.get(f"https://api.openalex.org/works/doi:{doi}", timeout=7)
        if r.status_code == 200:
            return r.json()
    except:
        pass
    return None

@lru_cache(maxsize=256)
def fetch_semantic(doi):
    try:
        r = requests.get(
            f"https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}",
            params={"fields": "abstract,tldr"},
            timeout=7
        )
        if r.status_code == 200:
            return r.json()
    except:
        pass
    return None

@lru_cache(maxsize=256)
def fetch_unpaywall(doi):
    """Fetch from Unpaywall API - often has better metadata"""
    try:
        r = requests.get(
            f"https://api.unpaywall.org/v2/{doi}",
            params={"email": "research@university.edu"},  # Required by Unpaywall
            timeout=7
        )
        if r.status_code == 200:
            return r.json()
    except:
        pass
    return None

# ==========================================================
# 📊 Impact Factor Lookup
# ==========================================================
def get_if_quartile(issn_list):
    for iss in issn_list:
        match = impact_df[
            (impact_df["ISSN"] == iss) | (impact_df["eISSN"] == iss)
        ]
        if len(match) > 0:
            row = match.iloc[0]
            return row["JIF 2024"], row["JIF Quartile"]
    return None, None

# ==========================================================
# 🟦 HEC Category Lookup
# ==========================================================
def get_hec_category(issn_list):
    for iss in issn_list:
        match = hec_df[hec_df["ISSN"] == iss]
        if len(match) > 0:
            return match.iloc[0]["HEC Category"]
    return "W"

# ==========================================================
# 🔥 MASTER METADATA BUILDER
# ==========================================================
def fetch_metadata(doi_raw):

    doi = extract_doi(doi_raw)

    cross = fetch_crossref(doi)

    if not cross:
        return {"Error": f"DOI not found → {doi}"}

    # -------- BASIC FIELDS --------
    title = cross.get("title", [None])[0]
    journal = cross.get("container-title", [None])[0]
    issn_list = cross.get("ISSN", [])

    # -------- PUBLICATION DATE (IMPROVED) --------
    # Try multiple date fields in Crossref
    pub_date = None

    # First try 'published' (online publication date)
    if cross.get("published"):
        pub_date = format_publication_date(cross["published"].get("date-parts"))

    # If not found, try 'published-print' (print publication date)
    if not pub_date and cross.get("published-print"):
        pub_date = format_publication_date(cross["published-print"].get("date-parts"))

    # If not found, try 'issued' (generic issue date)
    if not pub_date and cross.get("issued"):
        pub_date = format_publication_date(cross["issued"].get("date-parts"))

    # Extract year for APA citation (always need at least the year)
    year = None
    if cross.get("issued", {}).get("date-parts"):
        year = cross["issued"]["date-parts"][0][0]
    elif cross.get("published", {}).get("date-parts"):
        year = cross["published"]["date-parts"][0][0]

    pub_date = pub_date or "Not Available"

    volume = cross.get("volume")
    issue = cross.get("issue")
    pages = cross.get("page")

    # -------- Authors (CROSSREF ONLY) --------
    authors = []
    affs = set()

    if "author" in cross:
        for a in cross["author"]:
            name = f"{a.get('given', '')} {a.get('family', '')}".strip()
            authors.append(name)

            for af in a.get("affiliation", []):
                name = af.get("name", "").strip()
                if name:
                    affs.add(name)

    # -------- ABSTRACT (Multiple sources - ORIGINAL ONLY) --------
    abstract = None

    # Try 1: Crossref
    if cross.get("abstract"):
        abstract = clean_html(cross.get("abstract"))

    # Try 2: Semantic Scholar (full abstract ONLY - skip TLDR)
    if not abstract:
        s2 = fetch_semantic(doi)
        if s2 and s2.get("abstract"):
            abstract = s2["abstract"]

    # Try 3: Unpaywall
    if not abstract:
        unpaywall = fetch_unpaywall(doi)
        if unpaywall and unpaywall.get("abstract"):
            abstract = clean_html(unpaywall.get("abstract"))

    # Try 4: OpenAlex (sometimes has abstracts)
    if not abstract:
        oa = fetch_openalex(doi)
        if oa and oa.get("abstract_inverted_index"):
            # OpenAlex stores abstracts in inverted index format - reconstruct it
            try:
                inv_index = oa["abstract_inverted_index"]
                words = []
                for word, positions in inv_index.items():
                    for pos in positions:
                        words.append((pos, word))
                words.sort()
                abstract = " ".join([w[1] for w in words])
            except:
                pass

    abstract = abstract or "Not Available"

    # -------- KEYWORDS --------
    keywords = []

    if cross.get("subject"):
        keywords += cross["subject"]

    # Reuse OpenAlex call if we already fetched it for abstract
    if 'oa' not in locals():
        oa = fetch_openalex(doi)

    # accept OA only if DOI matches
    if oa and oa.get("doi", "").lower().replace("https://doi.org/", "") == doi.lower():
        for c in oa.get("concepts", [])[:5]:
            keywords.append(c["display_name"])

    keywords = ", ".join(list(dict.fromkeys(keywords))) if keywords else "Not Available"

    # -------- IF + HEC --------
    jif, quartile = get_if_quartile(issn_list)
    hec_cat = get_hec_category(issn_list)

    # -------- APA --------
    lead = authors[0] if authors else "Unknown"
    apa = f"{lead} et al. ({year}). {title}. {journal}, {volume}({issue}), {pages}. https://doi.org/{doi}"

    return {
        "DOI": doi,
        "Title": title,
        "Authors": ", ".join(authors),
        "Affiliations": ", ".join(list(affs)) if affs else "Not Available",
        "Journal": journal,
        "ISSN": ", ".join(issn_list),
        "Volume": volume,
        "Issue": issue,
        "Pages": pages,
        "Publication Date": pub_date,
        "Year": year,
        "Abstract": abstract,
        "Keywords": keywords,
        "Impact Factor": jif if jif else "Not Found",
        "JIF Quartile": quartile if quartile else "Not Found",
        "HEC Category": hec_cat,
        "APA Citation": apa
    }

# ==========================================================
# EXCEL EXPORT
# ==========================================================
def export_to_excel(all_data, role):
    df = pd.DataFrame(all_data)
    df["Submitted By"] = role
    df["Fetched On"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    df.to_excel(export_path, index=False)
    return export_path

# ==========================================================
# GRADIO UI WITH ROLE-BASED DISPLAY
# ==========================================================
def process(dois_text, role):
    dois = [d.strip() for d in re.split(r"[,\n]", dois_text) if d.strip()]
    results, html = [], ""

    # Define admin-only fields
    admin_only_fields = ["Impact Factor", "JIF Quartile", "HEC Category", "APA Citation"]

    for d in dois:
        data = fetch_metadata(d)
        results.append(data)

        html += "<div style='padding:10px;border:1px solid #ccc;margin:10px;'>"
        for k, v in data.items():
            # Skip admin-only fields if user is not admin
            if role != "admin" and k in admin_only_fields:
                continue
            html += f"<b>{k}:</b> {v}<br>"
        html += "</div>"

    file = export_to_excel(results, role)
    return html, file

gr.Interface(
    fn=process,
    inputs=[
        gr.Textbox(lines=6, label="Enter DOI(s)"),
        gr.Dropdown(["admin","user"], value="admin", label="Role")
    ],
    outputs=[gr.HTML(), gr.File()],
    title="DOI Metadata Extractor (Updated 10-12-25)"
).launch()

Available columns in HEC Excel: ['Title of the Journal', 'ISSN P', 'ISSN E', 'Publisher', 'Subject', 'Subject SubCategory', 'Category', 'From_Year', 'To_Year']
Loaded 1440 ISSN entries from HEC file
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57d51fceac8191a565.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
